# 12｜月底早尖峰低估案例
沿用固定的訓練與評估切分。以歷史平均低估量減去上一小時低估量，找出差距集中在哪些站與日期。

In [1]:
from zipfile import ZipFile
import pandas as pd
start, split, end = map(pd.Timestamp, ['2025-01-01', '2025-01-18', '2025-02-01'])
with ZipFile('../data/raw/202501-capitalbikeshare-tripdata.zip') as z:
    with z.open('202501-capitalbikeshare-tripdata.csv') as f:
        rides = pd.read_csv(f, usecols=['started_at', 'start_station_id', 'rideable_type'], dtype='string')
rides = rides.loc[rides.rideable_type.eq("classic_bike")].copy()
hour = pd.to_datetime(rides.started_at, format='mixed', errors='coerce').dt.floor('h')
station = rides.start_station_id.str.strip().replace('', pd.NA)
events = pd.DataFrame({'station': station, 'hour': hour})
events = events.loc[hour.ge(start) & hour.lt(end) & station.notna()]
stations = sorted(events.loc[events.hour.lt(split) & ~events.hour.dt.day.isin([1, 9]), 'station'].unique())
cohort = events[events.station.isin(stations)]
grid = pd.MultiIndex.from_product(
    [stations, pd.date_range(start, end, freq='h', inclusive='left')],
    names=['station', 'hour'])
panel = cohort.groupby(['station', 'hour']).size().reindex(grid, fill_value=0).rename('actual').to_frame()
assert panel.actual.sum() == len(cohort)


In [2]:
p = panel.reset_index()
p['lag'] = p.groupby('station').actual.shift(1)
p['date'] = p.hour.dt.normalize()
p['h'] = p.hour.dt.hour
p['weekend'] = p.hour.dt.dayofweek >= 5
ordinary = p.loc[~p.hour.dt.day.isin([1, 9, 20])]
train = ordinary.loc[ordinary.hour < split]
valid = ordinary.loc[ordinary.hour >= pd.Timestamp('2025-01-25')].copy()
keys = ['station', 'weekend', 'h']
valid = valid.join(train.groupby(keys).actual.mean().rename('history'), on=keys)
assert valid[['history', 'lag']].notna().all().all()
weekday = train.loc[~train.weekend]
def score(label, frame):
    row = [label, frame.station.nunique(), len(frame)]
    for pred in ['lag', 'history']:
        gap = frame.actual - frame[pred]
        row.extend([gap.abs().mean(), gap.clip(lower=0).mean()])
    return row
rows = [score('all', valid)]
for label, lo, hi in [('AM', 6, 10), ('PM', 15, 20)]:
    window = weekday.loc[weekday.h.between(lo, hi-1)]
    days = window.loc[window.actual.gt(0)].groupby('station').date.nunique()
    hourly = window.groupby(['station','h']).actual.mean().unstack()
    pairs = hourly.iloc[:, :-1] + hourly.shift(-1, axis=1).iloc[:, :-1]
    peak = pairs.idxmax(axis=1)
    for threshold in (5,):
        eligible = days[days.ge(threshold)].index
        v = valid.loc[~valid.weekend & valid.station.isin(eligible)]
        first = v.station.map(peak)
        rows.append(score(f'{label}/{threshold}', v.loc[(v.h == first) | (v.h == first + 1)]))
report = pd.DataFrame(rows, columns=['scope','stations','n','lag_MAE','lag_under','hist_MAE','hist_under'])
print(report.round(4).to_string(index=False))
print('Peak coverage at 5 days:')
print(report.loc[report.scope.isin(['AM/5','PM/5']), ['scope','stations']].assign(
    coverage=lambda d: d.stations / valid.station.nunique()).to_string(index=False))
am = weekday.loc[weekday.h.between(6,9)]
active = am.loc[am.actual.gt(0)].groupby('station').date.nunique()
example = active.loc[active.ge(5)].sort_index().index[0]
curve = am.loc[am.station.eq(example)].groupby('h').actual.mean()
print('Audit station:', example, 'active days:', active[example])
print(curve.round(4).to_string())
print('Chosen start:', (curve + curve.shift(-1)).idxmax())

am = weekday.loc[weekday.h.between(6,9)]
hours = am.groupby(['station','h']).actual.mean().unstack()
start_hour = (hours + hours.shift(-1,axis=1)).iloc[:,:-1].idxmax(axis=1)
eligible = am.loc[am.actual.gt(0)].groupby('station').date.nunique()
case = valid.loc[~valid.weekend & valid.station.isin(eligible[eligible.ge(5)].index)].copy()
first = case.station.map(start_hour)
case = case.loc[case.h.eq(first) | case.h.eq(first+1)].copy()
case['extra'] = (case.actual-case.history).clip(lower=0)-(case.actual-case.lag).clip(lower=0)
assert len(case)==2160
by_station=case.groupby('station').extra.sum().sort_values(ascending=False)
print('Stations positive / negative / tied:', by_station.gt(1e-9).sum(),by_station.lt(-1e-9).sum(),by_station.abs().le(1e-9).sum())
print('Net extra:',round(by_station.sum(),3),'Top five share of positive station contributions:',round(by_station.head().sum()/by_station.clip(lower=0).sum(),4))
print(case.groupby('date').extra.sum().round(3).to_string())
example=case.loc[case.station.eq(by_station.index[0])].sort_values('hour')
print(by_station.head().round(3).to_string())
print(example[['hour','actual','history','lag','extra']].round(3).to_string(index=False))


scope  stations      n  lag_MAE  lag_under  hist_MAE  hist_under
  all       712 119616   0.2838     0.1418    0.2258      0.1647
 AM/5       216   2160   1.4542     0.9194    1.1684      0.9682
 PM/5       322   3220   1.1040     0.6842    0.9962      0.7531
Peak coverage at 5 days:
scope  stations  coverage
 AM/5       216  0.303371
 PM/5       322  0.452247
Audit station: 30201 active days: 6
h
6    0.0000
7    0.2727
8    0.4545
9    0.0000
Chosen start: 7
Stations positive / negative / tied: 93 118 5
Net extra: 105.364 Top five share of positive station contributions: 0.2355
date
2025-01-27   -27.727
2025-01-28    64.818
2025-01-29    96.364
2025-01-30    22.545
2025-01-31   -50.636
station
31101    19.091
31324    13.636
31330    13.545
31600    13.273
31201    12.273
               hour  actual  history  lag  extra
2025-01-27 08:00:00       7    2.000  2.0  0.000
2025-01-27 09:00:00       5    1.636  7.0  3.364
2025-01-28 08:00:00      13    2.000  4.0  2.000
2025-01-28 09:00:00

低估量為 max(實際−預測, 0)，以所有入選格平均。零值沿用站點覆蓋假設。尖峰樣本因門檻而改變。